# Apply a persisted registration transform

This notebook applies the complete Cy-ExM forward chain to another intensity channel. The nonlinear warp is listed before the affine transform and neither is inverted.

In [ ]:
# EDIT THESE PATHS before running the notebook.
from pathlib import Path

FIXED_IMAGE = Path("path/to/fixed_reference.tif")
MOVING_IMAGE = Path("path/to/moving_channel.tif")
AFFINE_TRANSFORM = Path("path/to/linear_transform_2.mat")
NONLINEAR_WARP = Path("path/to/nonlinear_warp_2.nii.gz")
OUTPUT_IMAGE = Path("path/to/registered_channel.tif")
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
ITK_THREADS = 1
BLAS_THREADS = 1

`PIXEL_SIZE_XYZ` is in micrometers and `(x, y, z)` order. TIFF arrays remain `(z, y, x)`. Configure native threads before loading either image.

In [ ]:
from cyexm.io.images import load_ants_image
from cyexm.io.tiff import write_uint16_tiff
from cyexm.registration.transforms import transform_image
from cyexm.runtime import NativeThreadConfig, configure_native_threads

NATIVE_THREADS = NativeThreadConfig(ITK_THREADS, BLAS_THREADS)
configure_native_threads(NATIVE_THREADS)
fixed = load_ants_image(FIXED_IMAGE, PIXEL_SIZE_XYZ)
moving = load_ants_image(MOVING_IMAGE, PIXEL_SIZE_XYZ)

In [ ]:
registered = transform_image(
    fixed=fixed,
    moving=moving,
    affine_path=AFFINE_TRANSFORM,
    warp_path=NONLINEAR_WARP,
    interpolation="linear",
    native_threads=NATIVE_THREADS,
)
source_values = moving.numpy()
write_uint16_tiff(
    registered,
    OUTPUT_IMAGE,
    float(source_values.min()),
    float(source_values.max()),
)
print(f"Registered image written to {OUTPUT_IMAGE}")